In [36]:
from linearmodels.asset_pricing import LinearFactorModel
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm

#Data Loading
ff = pd.read_csv('F-F_Research_Data_Factors.csv', skiprows = 3)
ff=ff.rename(columns = {'Unnamed: 0': 'Date'})
ff = ff[ff['Date'].astype(str).str.match(r'^\d{6}$')]
ff['Date'] = pd.to_datetime(ff['Date'], format='%Y%m')
ff = ff.set_index('Date')
ff[['Mkt-RF', 'SMB', 'HML', 'RF']] = ff[['Mkt-RF', 'SMB', 'HML', 'RF']].astype(float) / 100

pp = pd.read_csv('25_Portfolios_5x5.csv', skiprows = 15)
pp = pp.rename(columns= {'Unnamed: 0': 'Date'})
pp = pp.iloc[:1198]
pp = pp[pp['Date'].astype(str).str.match(r'^\d{6}$')]
pp['Date']= pd.to_datetime(pp['Date'], format= '%Y%m')
pp = pp.set_index('Date')
pp = pp.astype(float) / 100

#Series/DataFrames
portfolios = []
alphas = []
alpha_t = []
betas_mkt = []
betas_SMB = []
betas_HML = []
R_squared = []
CAPM_alpha = []
CAPM_Rsquared = []

#FF3 loop
for port in pp.columns:
    y = pp[port] - ff['RF']
    X = ff[['Mkt-RF', 'SMB', 'HML']]
    X = sm.add_constant(X)

    data = pd.concat([y, X], axis = 1).dropna()
    y = data.iloc[:, 0]
    X = data.iloc[:, 1:]

    regression = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': 6})
    
    alphas.append(regression.params['const'])
    alpha_t.append(regression.tvalues['const'])
    betas_mkt.append(regression.params['Mkt-RF'])
    betas_SMB.append(regression.params['SMB'])
    betas_HML.append(regression.params['HML'])
    R_squared.append(regression.rsquared)
    portfolios.append(port)

#CAPM loop
for port in pp.columns:
    y_1 = pp[port] - ff['RF']
    X_1 = ff['Mkt-RF']
    X_1 = sm.add_constant(X_1)
    data_1 = pd.concat([y_1, X_1], axis = 1).dropna()
    y_1 = data_1.iloc[:, 0]
    X_1 = data_1.iloc[:, 1:]
    CAPM_reg = sm.OLS(y_1, X_1).fit(cov_type='HAC', cov_kwds={'maxlags': 6})

    CAPM_alpha.append(CAPM_reg.params['const'])  
    CAPM_Rsquared.append(CAPM_reg.rsquared)

#DataFrame construction
results_df = pd.DataFrame({'Portfolios': portfolios,
                           'Alpha': alphas,
                           'Alpha t': alpha_t,
                           'Beta Mkt-RF': betas_mkt,
                           'Beta SMB': betas_SMB,
                           'Beta HML': betas_HML,
                           'R squared': R_squared,
                          'CAPM alphas': CAPM_alpha,
                          'CAPM R squared': CAPM_Rsquared,
                          })

#Outputs
print(results_df.round(3))
CAPM_Rsquared = pd.Series(CAPM_Rsquared)
R_squared = pd.Series(R_squared)
print(f"FF3 Mean R squared: {R_squared.mean(): .3f}")
print(f"CAPM Mean R squared: {CAPM_Rsquared.mean(): .3f}")

#GRS test
excess_return = pp.sub(ff['RF'], axis=0)
combined = pd.concat([excess_return, ff[['Mkt-RF', 'SMB', 'HML']]], axis=1).dropna()
port_excess = combined.iloc[:, :25]
factors_clean = combined.iloc[:, 25:]
FF3_model = LinearFactorModel(port_excess, factors_clean).fit()
print(FF3_model)
print()

#GRS CAPM test
excess_return = pp.sub(ff['RF'], axis=0)
combined = pd.concat([excess_return, ff[['Mkt-RF']]], axis=1).dropna()
port_excess = combined.iloc[:, :25]
factors_clean = combined.iloc[:, 25:]
CAPM_GRS_model = LinearFactorModel(port_excess, factors_clean).fit()
print(CAPM_GRS_model)

#Date diagnostic
print(ff.index.min(), "to", ff.index.max())
print(pp.index.min(), "to", pp.index.max())


# Comparing the mean of absolute values
print(f"Mean |FF3 alpha|:  {np.mean(np.abs(alphas)):.5f}")
print(f"Mean |CAPM alpha|: {np.mean(np.abs(CAPM_alpha)):.5f}")

    Portfolios  Alpha  Alpha t  Beta Mkt-RF  Beta SMB  Beta HML  R squared  \
0   SMALL LoBM -0.007   -4.710        1.270     1.474     0.349      0.663   
1      ME1 BM2 -0.004   -3.580        1.071     1.529     0.206      0.823   
2      ME1 BM3 -0.001   -1.979        1.049     1.238     0.493      0.889   
3      ME1 BM4  0.001    1.283        0.941     1.233     0.576      0.929   
4   SMALL HiBM  0.001    2.420        0.981     1.286     0.888      0.929   
5      ME2 BM1 -0.002   -3.344        1.088     1.151    -0.230      0.909   
6      ME2 BM2  0.000    0.424        1.024     1.004     0.117      0.934   
7      ME2 BM3  0.000    0.486        0.992     0.831     0.343      0.934   
8      ME2 BM4  0.000    0.573        0.968     0.829     0.585      0.951   
9      ME2 BM5  0.000    0.683        1.064     0.927     0.885      0.952   
10     ME3 BM1 -0.001   -2.262        1.126     0.810    -0.220      0.926   
11     ME3 BM2  0.001    1.785        1.017     0.561     0.048 